# Multi-Feature Logistic Regression (Hyperplanes in $\mathbb{R}^n$)

When extending Logistic Regression from 2D space ($n=2$) to $n$ input features ($n > 2$), the linear combination becomes a dot product of weight and feature vectors:

$$z = \mathbf{w}^T \mathbf{x} + b = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b$$

### 1. Vectorized Sigmoid Hypothesis
The predicted probability $\hat{y} = P(y=1 \mid \mathbf{x})$ across a batch of $m$ samples ($\mathbf{X} \in \mathbb{R}^{m \times n}$) is:

$$\mathbf{\hat{Y}} = \sigma(\mathbf{X}\mathbf{w} + b) = \frac{1}{1 + e^{-(\mathbf{X}\mathbf{w} + b)}}$$

---

### 2. Decision Boundary: The $(n-1)$-Dimensional Hyperplane
The decision threshold $\hat{y} = 0.5$ occurs where $z = 0$:

$$\mathbf{w}^T \mathbf{x} + b = 0 \implies w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b = 0$$

In higher dimensions, this equation no longer defines a 2D line, but an **$(n-1)$-dimensional hyperplane** dividing the feature space into two regions:
- **Above Hyperplane ($z > 0$):** $\hat{y} \ge 0.5 \implies \text{Class 1}$
- **Below Hyperplane ($z < 0$):** $\hat{y} < 0.5 \implies \text{Class 0}$

---

## 3. Cost Function

### Unregularized Binary Cross-Entropy (Log Loss)
Measures the divergence between true binary labels $y^{(i)} \in \{0, 1\}$ and predicted probabilities $\hat{y}^{(i)} \in (0, 1)$:

$$J(\mathbf{w}, b) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{y}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)}) \right]$$

### Regularized Log Loss (L2 Ridge Penalty)
To penalize large weights and prevent weight explosion in higher dimensions, we add an L2 penalty term ($\frac{\lambda}{2m} \|\mathbf{w}\|^2$):

$$J(\mathbf{w}, b) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{y}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)}) \right] + \frac{\lambda}{2m} \sum_{j=1}^{n} w_j^2$$

Where:
- $\lambda \ge 0$ is the regularization hyperparameter.
- Note: The bias term $b$ is **never** penalized to avoid shifting the hyperplane origin artificially.

---

## 4. Gradients (Partial Derivatives)

Using the chain rule, the derivatives of the regularized cost function $J$ with respect to parameters $\mathbf{w}$ and $b$ are:

### Weight Gradients ($\mathbf{dw} \in \mathbb{R}^n$):
$$\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{m} \mathbf{X}^T (\mathbf{\hat{Y}} - \mathbf{Y}) + \frac{\lambda}{m}\mathbf{w}$$

### Bias Gradient ($db \in \mathbb{R}$):
$$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})$$

---

## 5. Parameter Update Rules (Gradient Descent)

At each training iteration/epoch, parameters are simultaneously updated using learning rate $\alpha$:

$$\mathbf{w} := \mathbf{w} - \alpha \frac{\partial J}{\partial \mathbf{w}} = \mathbf{w} - \alpha \left( \frac{1}{m} \mathbf{X}^T (\mathbf{\hat{Y}} - \mathbf{Y}) + \frac{\lambda}{m}\mathbf{w} \right)$$

$$b := b - \alpha \frac{\partial J}{\partial b} = b - \alpha \left( \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)}) \right)$$

### Explicit Weight Decay Formulation:
Factoring out $\mathbf{w}$ in the weight update step highlights the **weight decay** effect introduced by L2 regularization:

$$\mathbf{w} := \mathbf{w}\left(1 - \frac{\alpha \lambda}{m}\right) - \frac{\alpha}{m} \mathbf{X}^T (\mathbf{\hat{Y}} - \mathbf{Y})$$

*The factor $\left(1 - \frac{\alpha \lambda}{m}\right) < 1$ shrinks the weights slightly before subtracting the gradient step, capping weight magnitudes.*

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

class MyMultiFeatureLogisticRegression:
    def __init__(self, learning_rate=0.1, epochs=2000, l2_lambda=0.1):
        self.lr = learning_rate
        self.epochs = epochs
        self.l2_lambda = l2_lambda
        self.weights = None
        self.bias = None
        self.cost_history = []

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)  # Prevent numerical overflow in exp(-z)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X, y):
        m, n = X.shape
        self.weights = np.zeros(n)
        self.bias = 0.0

        for epoch in range(self.epochs):
            # 1. Forward Pass (Vectorized across all n features)
            linear_model = np.dot(X, self.weights) + self.bias
            y_pred_prob = self._sigmoid(linear_model)

            # 2. Regularized Binary Cross-Entropy Cost
            eps = 1e-15
            y_pred_clipped = np.clip(y_pred_prob, eps, 1 - eps)
            unreg_cost = - (1 / m) * np.sum(y * np.log(y_pred_clipped) + (1 - y) * np.log(1 - y_pred_clipped))
            l2_cost = (self.l2_lambda / (2 * m)) * np.sum(self.weights ** 2)
            cost = unreg_cost + l2_cost
            self.cost_history.append(cost)

            # 3. Compute Gradients with L2 Penalty
            dw = (1 / m) * np.dot(X.T, (y_pred_prob - y)) + (self.l2_lambda / m) * self.weights
            db = (1 / m) * np.sum(y_pred_prob - y)

            # 4. Parameter Updates
            self.weights -= self.lr * dw
            self.bias -= self.lr * db

            if epoch % (self.epochs // 10) == 0 or epoch == self.epochs - 1:
                print(f"  ➜ Epoch {epoch:4d} | Log Loss: {cost:.6f}")

    def predict_proba(self, X):
        return self._sigmoid(np.dot(X, self.weights) + self.bias)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

In [3]:
# --- Usage Section ---

# Data Setup  (Synthetic Data for demonstration)

np.random.seed(42)

# 1. Define True Ground-Truth Parameters in 6-Dimensional Space
true_w_6d = np.array([2.5, -1.8, 0.8, -3.2, 1.1, -0.5])
true_b_6d = 0.7

print("=" * 60)
print(" GROUND TRUTH PARAMETERS (6D HYPERPLANE)")
print("=" * 60)
for i, w in enumerate(true_w_6d):
    print(f" True Feature w_{i+1} : {w:6.2f}")
print(f" True Bias b     : {true_b_6d:6.2f}")

# 2. Generate 6D Features (m = 1000 samples)
m_samples = 1000
X_6d = np.random.randn(m_samples, 6)

# 3. Generate Labels y with moderate noise to simulate real-world overlap
z_6d = np.dot(X_6d, true_w_6d) + true_b_6d
noise_6d = np.random.normal(0, 0.4, size=m_samples)
probs_6d = 1.0 / (1.0 + np.exp(-(z_6d + noise_6d)))
y_6d = (probs_6d >= 0.5).astype(int)

# 4. Train Custom Gradient Descent Model
print("\n" + "=" * 60)
print(" 1. TRAINING CUSTOM MULTI-FEATURE GD MODEL")
print("=" * 60)
scratch_6d = MyMultiFeatureLogisticRegression(learning_rate=0.1, epochs=2000, l2_lambda=0.1)
scratch_6d.fit(X_6d, y_6d)

# 5. Train Scikit-Learn Model (C = 10.0 corresponding to l2_lambda = 0.1)
print("\n" + "=" * 60)
print(" 2. TRAINING SCIKIT-LEARN MODEL (L-BFGS)")
print("=" * 60)
sk_6d = LogisticRegression(C=10.0, solver='lbfgs')
sk_6d.fit(X_6d, y_6d);
print("✓ Scikit-Learn Training Complete!")

 GROUND TRUTH PARAMETERS (6D HYPERPLANE)
 True Feature w_1 :   2.50
 True Feature w_2 :  -1.80
 True Feature w_3 :   0.80
 True Feature w_4 :  -3.20
 True Feature w_5 :   1.10
 True Feature w_6 :  -0.50
 True Bias b     :   0.70

 1. TRAINING CUSTOM MULTI-FEATURE GD MODEL
  ➜ Epoch    0 | Log Loss: 0.693147
  ➜ Epoch  200 | Log Loss: 0.220950
  ➜ Epoch  400 | Log Loss: 0.174487
  ➜ Epoch  600 | Log Loss: 0.153364
  ➜ Epoch  800 | Log Loss: 0.140598
  ➜ Epoch 1000 | Log Loss: 0.131811
  ➜ Epoch 1200 | Log Loss: 0.125290
  ➜ Epoch 1400 | Log Loss: 0.120206
  ➜ Epoch 1600 | Log Loss: 0.116104
  ➜ Epoch 1800 | Log Loss: 0.112706
  ➜ Epoch 1999 | Log Loss: 0.109850

 2. TRAINING SCIKIT-LEARN MODEL (L-BFGS)
✓ Scikit-Learn Training Complete!


In [4]:
# 1. Compare Learned Weights across all 6 Dimensions
feature_labels = [f"w_{i+1} (Feature {i+1})" for i in range(6)] + ["Bias (b)"]

weights_comp_df = pd.DataFrame({
    "Parameter": feature_labels,
    "Ground Truth": list(true_w_6d) + [true_b_6d],
    "Custom GD Model": list(scratch_6d.weights) + [scratch_6d.bias],
    "Scikit-Learn Model": list(sk_6d.coef_[0]) + [sk_6d.intercept_[0]]
})

print("=" * 65)
print(" PARAMETER RECOVERY COMPARISON (6D HYPERPLANE)")
print("=" * 65)
print(weights_comp_df.to_string(index=False))

# 2. Compare Classification Metrics
y_pred_scratch = scratch_6d.predict(X_6d)
y_pred_sk = sk_6d.predict(X_6d)

metrics_6d_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Custom GD Model": [
        accuracy_score(y_6d, y_pred_scratch),
        precision_score(y_6d, y_pred_scratch),
        recall_score(y_6d, y_pred_scratch),
        f1_score(y_6d, y_pred_scratch)
    ],
    "Scikit-Learn Model": [
        accuracy_score(y_6d, y_pred_sk),
        precision_score(y_6d, y_pred_sk),
        recall_score(y_6d, y_pred_sk),
        f1_score(y_6d, y_pred_sk)
    ]
})

print("\n" + "=" * 65)
print(" METRICS COMPARISON ON 6D DATASET")
print("=" * 65)
print(metrics_6d_df.to_string(index=False))

 PARAMETER RECOVERY COMPARISON (6D HYPERPLANE)
      Parameter  Ground Truth  Custom GD Model  Scikit-Learn Model
w_1 (Feature 1)           2.5         3.472671            7.969181
w_2 (Feature 2)          -1.8        -2.533445           -5.826351
w_3 (Feature 3)           0.8         1.018655            2.389397
w_4 (Feature 4)          -3.2        -4.548325          -10.263594
w_5 (Feature 5)           1.1         1.485816            3.451537
w_6 (Feature 6)          -0.5        -0.852228           -1.888227
       Bias (b)           0.7         1.005441            2.295448

 METRICS COMPARISON ON 6D DATASET
   Metric  Custom GD Model  Scikit-Learn Model
 Accuracy         0.972000            0.975000
Precision         0.975352            0.978836
   Recall         0.975352            0.977113
 F1 Score         0.975352            0.977974


# Real-World Applications & Practical Model Alignment

To validate our Custom Gradient Descent Logistic Regression against industrial standards, we evaluate performance on two contrasting real-world domains with distinct dataset scales and preprocessing demands.

---

## Case Study 1: Clinical Tumor Classification (Breast Cancer Wisconsin)
- **Task:** Binary Diagnosis ($1 = \text{Malignant}$, $0 = \text{Benign}$)
- **Scale:** Small Dataset ($m = 569$, $n = 30$)
- **Preprocessing:** Z-score Feature Standardization ($\mu=0, \sigma=1$)

## Case Study 2: Adult Census Income Prediction
- **Task:** Income Level ($1 = >\$50\text{K/year}$, $0 = \le\$50\text{K/year}$)
- **Scale:** Large Dataset ($m \approx 48,842$, $n \approx 100+$ post-encoding)
- **Preprocessing:** Handling missing values, One-Hot Encoding for categorical variables, and Feature Standardization for continuous attributes.

---



## **1: Breast Cancer Wisconsin Dataset**. 

### Dataset Overview
- **Task:** Binary Classification (Malignant vs. Benign)
- **Features ($n = 30$):** Mean, standard error, and worst values of cell nucleus characteristics computed from digitized Fine Needle Aspirate (FNA) images (e.g., radius, texture, perimeter, area, smoothness, concavity).
- **Target ($y$):** 
  - $1 = \text{Malignant}$
  - $0 = \text{Benign}$

### Machine Learning Workflow for real-world applications
1. **Feature Scaling (Standardization):** Logistic regression relies on gradient updates ($\mathbf{X}^T (\mathbf{\hat{Y}} - \mathbf{Y})$). Features measured on wildly different scales (e.g., `area` $\approx 1000$ vs. `smoothness` $\approx 0.1$) distort gradient updates, making feature scaling essential.
2. **Train/Test Splitting:** Evaluate model generalization on unseen patient data.
3. **Regularization ($\text{L2}$):** Prevents overfitting across 30 feature dimensions.

In [13]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load Real-World Dataset
cancer_data = load_breast_cancer()
X_raw = cancer_data.data
y_raw = cancer_data.target  # 1 = Benign, 0 = Malignant in original sklearn dataset

# Re-map so 1 = Malignant (Positive Class) and 0 = Benign
y = np.where(y_raw == 0, 1, 0)
feature_names = cancer_data.feature_names

# 2. Train / Test Split (80% Train, 20% Test)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Feature Scaling (Z-score Standardization: mean=0, std=1)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print(f"Dataset Loaded Successfully:")
print(f" ➜ Training Samples : {X_train.shape[0]}")
print(f" ➜ Testing Samples  : {X_test.shape[0]}")
print(f" ➜ Number of Features: {X_train.shape[1]}")
print(f" ➜ Positive Cases (Malignant in Test Set): {np.sum(y_test)} / {len(y_test)}")

Dataset Loaded Successfully:
 ➜ Training Samples : 455
 ➜ Testing Samples  : 114
 ➜ Number of Features: 30
 ➜ Positive Cases (Malignant in Test Set): 42 / 114


In [14]:
# 1. Train Custom GD Logistic Regression with L2 Regularization
custom_model = MyMultiFeatureLogisticRegression(learning_rate=0.5, epochs=1000, l2_lambda=1.0)

print("=" * 60)
print(" TRAINING CUSTOM MODEL ON CLINICAL DATA")
print("=" * 60)
custom_model.fit(X_train, y_train)

# 2. Train Scikit-Learn Model
sk_model = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
sk_model.fit(X_train, y_train);

# 3. Evaluate Predictions on Test Set
y_pred_custom = custom_model.predict(X_test)
y_pred_sk = sk_model.predict(X_test)

# Metric Table
real_world_metrics = pd.DataFrame({
    "Evaluation Metric": ["Accuracy", "Precision", "Recall (Sensitivity)", "F1 Score"],
    "Custom GD Model": [
        accuracy_score(y_test, y_pred_custom),
        precision_score(y_test, y_pred_custom),
        recall_score(y_test, y_pred_custom),
        f1_score(y_test, y_pred_custom)
    ],
    "Scikit-Learn Model": [
        accuracy_score(y_test, y_pred_sk),
        precision_score(y_test, y_pred_sk),
        recall_score(y_test, y_pred_sk),
        f1_score(y_test, y_pred_sk)
    ]
})

print("\n" + "=" * 65)
print(" PERFORMANCE ON UNSEEN CLINICAL TEST SET")
print("=" * 65)
print(real_world_metrics.to_string(index=False))

 TRAINING CUSTOM MODEL ON CLINICAL DATA
  ➜ Epoch    0 | Log Loss: 0.693147
  ➜ Epoch  100 | Log Loss: 0.077954
  ➜ Epoch  200 | Log Loss: 0.072813
  ➜ Epoch  300 | Log Loss: 0.070928
  ➜ Epoch  400 | Log Loss: 0.070021
  ➜ Epoch  500 | Log Loss: 0.069530
  ➜ Epoch  600 | Log Loss: 0.069242
  ➜ Epoch  700 | Log Loss: 0.069063
  ➜ Epoch  800 | Log Loss: 0.068946
  ➜ Epoch  900 | Log Loss: 0.068868
  ➜ Epoch  999 | Log Loss: 0.068815

 PERFORMANCE ON UNSEEN CLINICAL TEST SET
   Evaluation Metric  Custom GD Model  Scikit-Learn Model
            Accuracy         0.973684            0.964912
           Precision         1.000000            0.975000
Recall (Sensitivity)         0.928571            0.928571
            F1 Score         0.962963            0.951220


In [16]:
# Extract weights to analyze which clinical features contribute most to diagnosing Malignancy
weights_df = pd.DataFrame({
    "Clinical Feature": feature_names,
    "Learned Weight (w)": custom_model.weights,
    "Abs Weight": np.abs(custom_model.weights)
}).sort_values(by="Abs Weight", ascending=False).reset_index(drop=True)

print("=" * 65)
print(" TOP 10 MOST INFLUENTIAL CLINICAL FEATURES FOR DIAGNOSIS")
print("=" * 65)
print(weights_df[["Clinical Feature", "Learned Weight (w)"]].head(10).to_string(index=False))
print("\nInterpretation:")
print(" • Positive weights increase P(Malignant) — e.g., higher worst concavity or worst radius.")
print(" • Negative weights increase P(Benign).")

 TOP 10 MOST INFLUENTIAL CLINICAL FEATURES FOR DIAGNOSIS
   Clinical Feature  Learned Weight (w)
      worst texture            1.318342
       radius error            1.170126
     worst symmetry            1.025204
  compactness error           -0.886340
       worst radius            0.877950
mean concave points            0.874907
         worst area            0.864291
         area error            0.855833
    worst concavity            0.852531
    perimeter error            0.746277

Interpretation:
 • Positive weights increase P(Malignant) — e.g., higher worst concavity or worst radius.
 • Negative weights increase P(Benign).


## **2: Adult Census Income Dataset** 

### Dataset Overview
- **Task:** Binary Classification (High Income vs. Low Income)
- **Features ($n \approx 100+$ post-encoding):** Demographic and employment attributes extracted from 1994 US Census data, including continuous features (e.g., `age`, `capital-gain`, `capital-loss`, `hours-per-week`) and categorical attributes (e.g., `workclass`, `education`, `marital-status`, `occupation`, `relationship`, `race`, `sex`).
- **Target ($y$):** 
  - $1 = \text{Income} > \$50\text{K/year}$
  - $0 = \text{Income} \le \$50\text{K/year}$

### Machine Learning Workflow for Real-World Applications
1. **Preprocessing & Categorical Encoding:** Categorical string variables are transformed using One-Hot Encoding to make them suitable for linear model dot-product operations, expanding feature space into a sparse representation.
2. **Feature Scaling (Standardization):** Continuous features with wide numerical ranges (e.g., `capital-gain` up to $\$99,999$ vs. `age` around $30$) distort gradient steps ($\mathbf{X}^T (\mathbf{\hat{Y}} - \mathbf{Y})$). Standardizing continuous features ($\mu = 0, \sigma = 1$) ensures isotropic gradient updates and stable convergence.
3. **Train/Test Splitting:** A stratified $80/20$ train-test split preserves the natural class proportion (~$24\%$ high-income) to reliably measure generalization on unseen data.
4. **Regularization ($\text{L2}$):** Penalizes large weights to prevent overfitting across high-dimensional features resulting from one-hot encoding.

In [17]:
# [Loading a Real-World Large Dataset — Adult Census Income]
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import pandas as pd
import numpy as np

# Load Real-World Adult Census Dataset (~48,842 rows)
print("Fetching real-world dataset from OpenML (Adult Census Income)...")
adult = fetch_openml(name='adult', version=2, as_frame=True, parser='auto')
df_raw = adult.frame.dropna()

# Target: Income >50K (1) or <=50K (0)
X_raw = df_raw.drop(columns=['class'])
y = (df_raw['class'].astype(str).str.contains('>50K')).astype(int).values

# Preprocess Categorical & Numerical Features
num_cols = X_raw.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X_raw.select_dtypes(include=['category', 'object']).columns

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
])

X = preprocessor.fit_transform(X_raw)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=" * 60)
print(" REAL-WORLD DATASET LOADED (ADULT CENSUS)")
print("=" * 60)
print(f" ➜ Total Records    : {X.shape[0]:,}")
print(f" ➜ Encoded Features : {X.shape[1]}")
print(f" ➜ Train Samples    : {X_train.shape[0]:,}")
print(f" ➜ Test Samples     : {X_test.shape[0]:,}")

Fetching real-world dataset from OpenML (Adult Census Income)...
 REAL-WORLD DATASET LOADED (ADULT CENSUS)
 ➜ Total Records    : 45,222
 ➜ Encoded Features : 96
 ➜ Train Samples    : 36,177
 ➜ Test Samples     : 9,045


In [18]:
# 1. Train Custom GD Model
custom_adult = MyMultiFeatureLogisticRegression(learning_rate=0.1, epochs=2000, l2_lambda=1.0)

print("=" * 60)
print(" TRAINING CUSTOM MODEL ON REAL-WORLD CENSUS DATA")
print("=" * 60)
custom_adult.fit(X_train, y_train)

# 2. Train Scikit-Learn Model
sk_adult = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
sk_adult.fit(X_train, y_train);

# Evaluate
y_pred_custom = custom_adult.predict(X_test)
y_pred_sk = sk_adult.predict(X_test)

metrics_adult_df = pd.DataFrame({
    "Evaluation Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Custom GD Model": [
        accuracy_score(y_test, y_pred_custom),
        precision_score(y_test, y_pred_custom),
        recall_score(y_test, y_pred_custom),
        f1_score(y_test, y_pred_custom)
    ],
    "Scikit-Learn Model": [
        accuracy_score(y_test, y_pred_sk),
        precision_score(y_test, y_pred_sk),
        recall_score(y_test, y_pred_sk),
        f1_score(y_test, y_pred_sk)
    ]
})

print("\n" + "=" * 65)
print(" PERFORMANCE ON UNSEEN CENSUS TEST SET (~9,000 samples)")
print("=" * 65)
print(metrics_adult_df.to_string(index=False))

 TRAINING CUSTOM MODEL ON REAL-WORLD CENSUS DATA
  ➜ Epoch    0 | Log Loss: 0.693147
  ➜ Epoch  200 | Log Loss: 0.366888
  ➜ Epoch  400 | Log Loss: 0.350104
  ➜ Epoch  600 | Log Loss: 0.342985
  ➜ Epoch  800 | Log Loss: 0.338885
  ➜ Epoch 1000 | Log Loss: 0.336178
  ➜ Epoch 1200 | Log Loss: 0.334258
  ➜ Epoch 1400 | Log Loss: 0.332832
  ➜ Epoch 1600 | Log Loss: 0.331740
  ➜ Epoch 1800 | Log Loss: 0.330883
  ➜ Epoch 1999 | Log Loss: 0.330201

 PERFORMANCE ON UNSEEN CENSUS TEST SET (~9,000 samples)
Evaluation Metric  Custom GD Model  Scikit-Learn Model
         Accuracy         0.842123            0.845992
        Precision         0.731513            0.732094
           Recall         0.573595            0.597235
         F1 Score         0.643000            0.657824


## Understanding Performance Differences: Custom GD vs. Scikit-Learn

When comparing custom Gradient Descent against Scikit-Learn's `LogisticRegression`, you may observe slight performance variations on test sets. It is important to note that **neither model is inherently better or worse**; these discrepancies stem from mathematical optimization and stopping criteria settings:

### 1. Why Scikit-Learn may appear slightly lower in Case Study 1:
- **Regularization Formulations:** Scikit-Learn uses a parameter $C = \frac{1}{\lambda}$, where the loss function scales as $C \cdot \text{Loss} + \frac{1}{2}\Vert{}\mathbf{w}\Vert{}^2$. In contrast, custom implementations often scale the penalty as $\frac{\lambda}{2m}\Vert{}\mathbf{w}\Vert{}^2$.
- **Regularization Magnitude:** At default $C=1.0$, Scikit-Learn applies a slightly stronger penalty relative to sample size $m$, pulling weights tighter toward zero. On a small test set (~114 samples), this can flip a single borderline prediction, causing a $\approx 0.88\%$ shift in metrics.
- **Tuning Fix:** Matching $C$ to the exact custom penalty formulation ($\lambda/m$) yields identical decision boundaries.

### 2. Why Custom GD may appear slightly lower in Case Study 2:
- **Fixed Epochs & Learning Rate:** Large datasets with post-encoding sparse matrices require more optimization steps or adaptive learning rates to reach complete convergence. A fixed epoch count (e.g., 2,000 iterations) or fixed learning rate ($\alpha=0.1$) may stop short of the absolute loss minimum.
- **Solver Efficiency:** Scikit-Learn uses second-order quasi-Newton algorithms (`L-BFGS`), which approximate Hessian matrices to take optimal step sizes and converge faster on high-dimensional data.
- **Tuning Fix:** Increasing epochs, tuning $\alpha$, or implementing learning rate decay allows Custom GD to match Scikit-Learn's metrics exactly.